In [1]:
import numpy as np
import nibabel as nib
import pyvista as pv
from scipy.ndimage import label, center_of_mass
from scipy.spatial import Voronoi


def render_voronoi_basal_cells_3d(
    mask_nifti: str,
    *,
    min_voxels: int = 30,
    show_overall_surface: bool = True,
    overall_opacity: float = 0.08,
    cell_opacity: float = 0.90,
    window_size=(1200, 800),
    jitter: float = 1e-8,
    show_axes: bool = True,
    add_text: bool = True,
    title: str = "Basal = RED | Internal = BLUE",
):
    """
    Interactive 3D rendering of connected-component nuclei (full meshes),
    colored by Voronoi basal rule:
      - Basal (RED): unbounded Voronoi region (contains -1)
      - Internal (BLUE): bounded Voronoi region

    Input:
      mask_nifti: path to NIfTI mask (binary or labeled; any >0 treated as mask)

    Returns:
      dict with keys: plotter, basal_flags, points_xyz, labeled_zyx, spacing_xyz
    """
    # ----------------------
    # Load NIfTI
    # ----------------------
    nii = nib.load(mask_nifti)
    data = nii.get_fdata()
    spacing_xyz = np.array(nii.header.get_zooms()[:3], dtype=float)  # (x,y,z)

    print("Volume shape:", data.shape)
    print("Voxel spacing (x,y,z):", spacing_xyz)

    # NIfTI array typically comes as (Z,Y,X)
    mask_zyx = (data > 0).astype(np.uint8)

    # ----------------------
    # Connected components
    # ----------------------
    labeled_zyx, num_cells = label(mask_zyx)
    print("Detected cells:", num_cells)
    if num_cells < 5:
        raise ValueError("Need >= 5 cells for stable 3D Voronoi classification.")

    # ----------------------
    # Centroids (z,y,x) -> world (x,y,z)
    # ----------------------
    centroids_zyx = np.array(
        center_of_mass(mask_zyx, labeled_zyx, range(1, num_cells + 1)),
        dtype=float,
    )

    points_xyz = np.zeros((len(centroids_zyx), 3), dtype=float)
    points_xyz[:, 0] = centroids_zyx[:, 2] * spacing_xyz[0]  # x
    points_xyz[:, 1] = centroids_zyx[:, 1] * spacing_xyz[1]  # y
    points_xyz[:, 2] = centroids_zyx[:, 0] * spacing_xyz[2]  # z

    # ----------------------
    # Voronoi basal flag
    # ----------------------
    # pts_j = points_xyz + jitter * np.random.randn(*points_xyz.shape)
    # vor = Voronoi(pts_j)

    # ----------------------
    # Voronoi basal flag (STRICT 3D ONLY)
    # ----------------------
    P = points_xyz - points_xyz.mean(axis=0, keepdims=True)
    rank = np.linalg.matrix_rank(P)
    print("Centroid point-cloud rank:", rank, "(3 means true 3D)")
    
    if rank < 3:
        raise ValueError(
            "Not true 3D: centroids are coplanar/near-coplanar (rank < 3). "
            "Voronoi becomes effectively 2D. Use thicker Z data or spread points in Z."
        )
    
    pts_j = points_xyz + jitter * np.random.randn(*points_xyz.shape)
    vor = Voronoi(pts_j)

    basal = np.zeros(len(points_xyz), dtype=bool)
    for i, region_index in enumerate(vor.point_region):
        region = vor.regions[region_index]
        basal[i] = (region is None) or (len(region) == 0) or (-1 in region)

    print("Basal:", int(basal.sum()), "Internal:", int((~basal).sum()))

    # ----------------------
    # Fix axis order for PyVista grid
    # PyVista ImageData dimensions expect (X,Y,Z)
    # Our label is (Z,Y,X) -> transpose
    # ----------------------
    labels_xyz = np.transpose(labeled_zyx, (2, 1, 0)).astype(np.int32)  # (X,Y,Z)
    mask_xyz = (labels_xyz > 0).astype(np.uint8)

    # ----------------------
    # Build overall surface (optional)
    # ----------------------
    overall_surface = None
    if show_overall_surface:
        grid_all = pv.ImageData()
        grid_all.dimensions = mask_xyz.shape
        grid_all.spacing = spacing_xyz
        grid_all.origin = (0, 0, 0)
        grid_all.point_data["mask"] = mask_xyz.flatten(order="F")
        overall_surface = grid_all.contour([0.5], scalars="mask")

    # ----------------------
    # Visualization: full nuclei meshes
    # ----------------------
    plotter = pv.Plotter(window_size=window_size)

    if show_overall_surface and overall_surface is not None:
        plotter.add_mesh(overall_surface, color="lightgray", opacity=overall_opacity)

    for cid in range(1, num_cells + 1):
        cell_bin = (labels_xyz == cid).astype(np.uint8)
        if int(cell_bin.sum()) < int(min_voxels):
            continue

        cell_grid = pv.ImageData()
        cell_grid.dimensions = cell_bin.shape
        cell_grid.spacing = spacing_xyz
        cell_grid.origin = (0, 0, 0)
        cell_grid.point_data["cell"] = cell_bin.flatten(order="F")

        cell_surf = cell_grid.contour([0.5], scalars="cell")

        color = "red" if basal[cid - 1] else "blue"
        plotter.add_mesh(cell_surf, color=color, opacity=cell_opacity)

    if add_text and title:
        plotter.add_text(title, font_size=12)

    if show_axes:
        plotter.show_axes()

    plotter.show()

    return {
        "plotter": plotter,
        "basal_flags": basal,
        "points_xyz": points_xyz,
        "labeled_zyx": labeled_zyx,
        "spacing_xyz": spacing_xyz,
    }


# ======================
# Example call
# ======================
# out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/test3_mask.nii.gz")

In [2]:


# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/all_data_for_model/mask/MCf7-1_mask.nii.gz")


Volume shape: (213, 177, 50)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 64
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 38 Internal: 26


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcd8802a640_0&reconnect=auto" class="pyvi…

In [3]:

# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/all_data_for_model/mask/10a_day5_a_mask.nii.gz")


Volume shape: (248, 248, 63)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 8
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 8 Internal: 0


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdbaf59fd0_1&reconnect=auto" class="pyvi…

In [4]:

# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/all_data_for_model/mask/AT1-1_mask.nii.gz")


Volume shape: (213, 177, 36)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 21
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 16 Internal: 5


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdd8aad7f0_2&reconnect=auto" class="pyvi…

In [5]:

# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/isotropic/MCF7/no_apotome_processed/MCF7_no_apotmome_processed_mask_iso.nii.gz")


Volume shape: (210, 177, 65)
Voxel spacing (x,y,z): [0.001 0.001 0.001]
Detected cells: 85
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 48 Internal: 37


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdc834b460_3&reconnect=auto" class="pyvi…

In [6]:

# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/test4_ring_all_basal_mask.nii.gz")


Volume shape: (128, 128, 48)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 16
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 16 Internal: 0


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdbc6f0730_4&reconnect=auto" class="pyvi…

In [7]:

# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/test6_shell_mask.nii.gz")


Volume shape: (128, 128, 64)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 27
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 23 Internal: 4


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdca151af0_5&reconnect=auto" class="pyvi…

In [8]:

# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/test7_two_shells_mask.nii.gz")


Volume shape: (140, 140, 80)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 50
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 28 Internal: 22


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdd896ffa0_6&reconnect=auto" class="pyvi…

In [9]:


# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/test8_mask.nii.gz")


Volume shape: (128, 128, 64)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 7
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 6 Internal: 1


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdd896f3d0_7&reconnect=auto" class="pyvi…

In [10]:



# ======================
# Example call
# ======================
out = render_voronoi_basal_cells_3d("/Users/muhammadsohaib/Downloads/all_data_for_model/mask/MCf7-1_mask.nii.gz")


Volume shape: (213, 177, 50)
Voxel spacing (x,y,z): [1. 1. 1.]
Detected cells: 64
Centroid point-cloud rank: 3 (3 means true 3D)
Basal: 38 Internal: 26


Widget(value='<iframe src="http://localhost:57168/index.html?ui=P_0x7fcdd9787d60_8&reconnect=auto" class="pyvi…

In [11]:
# ============================================================
# UPDATED COMPLETE CODE
# Voronoi basal classification (CSV only)
# Uses SAME logic as final 3D PyVista code
# Input: ONLY TWO FILES (mask + raw)
# Output: CSV with Basal / Non-basal labels
# Also prints:
#   - total cells
#   - basal cells
#   - non-basal cells
# Note:
#   - Basal / Non-basal is computed from MASK geometry only
#   - RAW is used only for per-cell intensity statistics
#   - CSV keeps only WORLD coordinates (x_world, y_world, z_world)
# ============================================================

import os
import numpy as np
import nibabel as nib
import pandas as pd

from scipy.ndimage import label, center_of_mass
from scipy.spatial import Voronoi


# ----------------------------
# Load NIfTI safely
# ----------------------------
def load_nii_3d(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    img = nib.load(path)
    arr = np.asanyarray(img.dataobj)

    if arr.ndim > 3:
        arr = np.squeeze(arr)
        if arr.ndim > 3:
            arr = arr[..., 0]

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume, got {arr.shape} from {path}")

    return arr, img


def sanitize_raw(raw: np.ndarray) -> np.ndarray:
    raw = raw.astype(np.float32, copy=False)
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    return raw


# ----------------------------
# EXACT SAME mask -> connected components logic
# as your final PyVista code
# ----------------------------
def get_connected_components_from_mask(mask: np.ndarray):
    mask_zyx = (mask > 0).astype(np.uint8)
    labeled_zyx, num_cells = label(mask_zyx)
    return mask_zyx, labeled_zyx, int(num_cells)


# ----------------------------
# EXACT SAME centroid logic as PyVista code
# plus intensity stats from raw
# ----------------------------
def compute_cell_df_from_labeled(
    mask_zyx: np.ndarray,
    labeled_zyx: np.ndarray,
    raw: np.ndarray,
    spacing_xyz: np.ndarray,
) -> pd.DataFrame:

    num_cells = int(labeled_zyx.max())
    if num_cells < 1:
        raise ValueError("No cells found in mask.")

    # SAME centroid logic as PyVista code
    # returns centroids in voxel coordinates (z,y,x)
    centroids_zyx = np.array(
        center_of_mass(mask_zyx, labeled_zyx, range(1, num_cells + 1)),
        dtype=float,
    )

    rows = []
    for cid in range(1, num_cells + 1):
        coords = np.argwhere(labeled_zyx == cid)
        if coords.size == 0:
            continue

        vals = raw[coords[:, 0], coords[:, 1], coords[:, 2]]

        cz, cy, cx = centroids_zyx[cid - 1]

        # SAME world conversion as PyVista code
        x = float(cx * spacing_xyz[0])
        y = float(cy * spacing_xyz[1])
        z = float(cz * spacing_xyz[2])

        rows.append({
            "cell_id": int(cid),
            "x": x,
            "y": y,
            "z": z,
            "nucleus_size": int(coords.shape[0]),
            "mean_intensity": float(np.mean(vals)),
            "median_intensity": float(np.median(vals)),
            "std_intensity": float(np.std(vals, ddof=0)),
        })

    df = pd.DataFrame(rows).sort_values("cell_id").reset_index(drop=True)

    if df.empty:
        raise ValueError("No cells found after connected components.")

    return df


# ----------------------------
# EXACT SAME Voronoi logic as PyVista code
# Basal = unbounded Voronoi region
# ----------------------------
def classify_basal_voronoi_exact(df: pd.DataFrame, jitter: float = 1e-8) -> pd.DataFrame:
    pts = df[["x", "y", "z"]].to_numpy(np.float64)

    if pts.shape[0] < 5:
        raise ValueError("Need >= 5 cells for stable 3D Voronoi classification.")

    # strict 3D check
    P = pts - pts.mean(axis=0, keepdims=True)
    rank = np.linalg.matrix_rank(P)
    print("Centroid point-cloud rank:", rank, "(3 means true 3D)")

    if rank < 3:
        raise ValueError(
            "Not true 3D: centroids are coplanar/near-coplanar (rank < 3). "
            "Voronoi becomes effectively 2D. Use thicker Z data or spread points in Z."
        )

    pts_j = pts + jitter * np.random.randn(*pts.shape)
    vor = Voronoi(pts_j)

    basal_flags = np.zeros(len(pts), dtype=bool)
    for i, region_index in enumerate(vor.point_region):
        region = vor.regions[region_index]
        basal_flags[i] = (region is None) or (len(region) == 0) or (-1 in region)

    out = df.copy()
    out["cell_type"] = np.where(basal_flags, "Basal", "Non-basal")
    out["is_basal"] = basal_flags.astype(int)
    return out


# ----------------------------
# Main pipeline
# ----------------------------
def run_voronoi_basal_pipeline(
    nuclei_mask_nifti: str,
    raw_dapi_nifti: str,
    out_csv: str = "Calculations_voronoi.csv",
    jitter: float = 1e-8,
) -> pd.DataFrame:

    mask, mask_img = load_nii_3d(nuclei_mask_nifti)
    raw, _ = load_nii_3d(raw_dapi_nifti)
    raw = sanitize_raw(raw)

    if mask.shape != raw.shape:
        raise ValueError(f"Shape mismatch: mask={mask.shape}, raw={raw.shape}")

    spacing_xyz = np.array(mask_img.header.get_zooms()[:3], dtype=float)

    # EXACT SAME connected-component logic as PyVista code
    mask_zyx, labeled_zyx, num_cells = get_connected_components_from_mask(mask)

    print("Volume shape      :", mask.shape)
    print("Detected cells    :", num_cells)

    if num_cells < 5:
        raise ValueError("Need >= 5 cells for stable 3D Voronoi classification.")

    df = compute_cell_df_from_labeled(
        mask_zyx=mask_zyx,
        labeled_zyx=labeled_zyx,
        raw=raw,
        spacing_xyz=spacing_xyz,
    )

    df = classify_basal_voronoi_exact(df, jitter=jitter)

    total_cells = len(df)
    total_basal = int((df["cell_type"] == "Basal").sum())
    total_non_basal = int((df["cell_type"] == "Non-basal").sum())

    print("Total cells      :", total_cells)
    print("Basal cells      :", total_basal)
    print("Non-basal cells  :", total_non_basal)

    df.to_csv(out_csv, index=False)
    print("Saved CSV:", out_csv)

    return df


# ============================================================
# EXAMPLE CALL
# ============================================================
if __name__ == "__main__":
    df = run_voronoi_basal_pipeline(
        nuclei_mask_nifti="/Users/muhammadsohaib/Downloads/all_data_for_model/mask/MCf7-1_mask.nii.gz",
        raw_dapi_nifti="/Users/muhammadsohaib/Downloads/all_data_for_model/image/MCf7-1.nii.gz",
        out_csv="Calculations_voronoi.csv",
     )

Volume shape      : (213, 177, 50)
Detected cells    : 64
Centroid point-cloud rank: 3 (3 means true 3D)
Total cells      : 64
Basal cells      : 38
Non-basal cells  : 26
Saved CSV: Calculations_voronoi.csv
